In [ ]:
from dotenv import load_dotenv
from databricks.sdk import WorkspaceClient
load_dotenv(override=True)
w = WorkspaceClient()
for e in w.serving_endpoints.list():
    print(e.name, "|", e.task) - 

In [ ]:
import os, re
from dotenv import load_dotenv
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

load_dotenv(override=True)
w = WorkspaceClient()

MODEL = "databricks-meta-llama-3-3-70b-instruct"
C, S = "workspace", "ai_data_team"
wh = next(iter(w.warehouses.list()))

def run(sql):
    r = w.statement_execution.execute_statement(
        statement=sql, warehouse_id=wh.id, wait_timeout="50s")
    if r.status.state.value != "SUCCEEDED":
        raise RuntimeError(r.status.error.message)
    return r

def llm(system, user, max_tokens=600):
    r = w.serving_endpoints.query(
        name=MODEL, max_tokens=max_tokens, temperature=0,
        messages=[ChatMessage(role=ChatMessageRole.SYSTEM, content=system),
                  ChatMessage(role=ChatMessageRole.USER, content=user)])
    return r.choices[0].message.content

# --- Tell the agent what data exists ---
cols = run(f"""SELECT table_name, column_name, data_type
               FROM {C}.information_schema.columns
               WHERE table_schema = '{S}' AND table_name LIKE 'clean_%'
               ORDER BY table_name, ordinal_position""").result.data_array
schema = {}
for t, c, d in cols:
    schema.setdefault(t, []).append(f"{c} ({d})")
SCHEMA_TEXT = "\n".join(f"{C}.{S}.{t}: " + ", ".join(cs) for t, cs in schema.items())

DATA_NOTES = """
- Amounts are in USD. Revenue = SUM(total_amount_usd) from clean_orders.
- Use clean_orders for revenue and order counts. Use order_date for anything over time.
- Do NOT use registration_date for tenure or signup cohorts: most orders predate it (data quirk).
- clean_monthly_revenue and clean_product_summary are pre-aggregated; prefer clean_orders for custom cuts.
"""

LATEST = run(f"SELECT MAX(order_date) FROM {C}.{S}.clean_orders").result.data_array[0][0]
DATA_NOTES += f"""
- The latest order in the data is {LATEST}. For relative periods ("last 12 months", "this year", "recently"), anchor to {LATEST}, never CURRENT_DATE. Example: last 12 months = order_date > DATE_ADD('{LATEST}', -365).
- "Best sellers" means highest SUM(quantity). Always include the ranking metric (units sold) and SUM(total_amount_usd) as columns, not just names.
"""

SQL_SYSTEM = f"""You write Databricks SQL for a business analyst agent.
Tables:
{SCHEMA_TEXT}
Notes:{DATA_NOTES}
Return ONLY one SELECT query, no explanation, no markdown."""

FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|create|alter|merge|truncate|grant|revoke)\b", re.I)

def write_sql(question):
    sql = llm(SQL_SYSTEM, question)
    sql = re.sub(r"^```(?:sql)?|```$", "", sql.strip(), flags=re.M).strip().rstrip(";")
    if not re.match(r"^(select|with)\b", sql, re.I) or FORBIDDEN.search(sql) or ";" in sql:
        raise ValueError(f"Blocked unsafe or invalid SQL:\n{sql}")
    return sql

def ask(question, show_sql=True):
    sql = write_sql(question)
    r = run(f"SELECT * FROM ({sql}) LIMIT 200")
    names = [c.name for c in r.manifest.schema.columns]
    rows = r.result.data_array if r.result and r.result.data_array else []
    if show_sql:
        print("SQL used:\n", sql, "\n")
    print(llm(
        "You explain data results to a non-technical business client. Use plain English, "
        "lead with the answer, quote the key numbers (USD), keep it under 120 words, "
        "and never invent numbers that are not in the results. If results are empty, say so.",
        f"Question: {question}\nColumns: {names}\nRows: {rows[:50]}\nData notes: {DATA_NOTES}"))

ask("What are the top 5 best-selling products in the last 12 months?")

In [ ]:
import json

def plan(question):
    out = llm(
        "You are a planner for a data team. Break the client's question into 1 to 3 short "
        "analytical sub-questions, each answerable with ONE SQL query on an orders/customers "
        "dataset. If the question is already simple, return it as the only item. "
        'Reply with ONLY a JSON list of strings, e.g. ["...", "..."].',
        question)
    try:
        subs = json.loads(re.search(r"\[.*\]", out, re.S).group(0))
        return [s for s in subs if isinstance(s, str)][:3] or [question]
    except Exception:
        return [question]

def analyst(subq, feedback=None):
    q = subq if not feedback else f"{subq}\nA reviewer rejected your previous SQL: {feedback}. Fix it."
    sql = write_sql(q)
    r = run(f"SELECT * FROM ({sql}) LIMIT 200")
    names = [c.name for c in r.manifest.schema.columns]
    rows = r.result.data_array if r.result and r.result.data_array else []
    return sql, names, rows

def reviewer(subq, sql, names, rows):
    verdict = llm(
        "You review SQL written for a data analyst. Check it against these rules:\n"
        f"{DATA_NOTES}\nAlso check that it actually answers the question. "
        "Reply with exactly 'OK' or 'FIX: <one sentence>'.",
        f"Question: {subq}\nSQL: {sql}\nColumns: {names}\nFirst rows: {rows[:5]}", max_tokens=100)
    return verdict.strip()

def team(question):
    print("QUESTION:", question)
    subs = plan(question)
    print("PLANNER ->", subs, "\n")
    findings = []
    for sq in subs:
        feedback = None
        for attempt in range(2):
            try:
                sql, names, rows = analyst(sq, feedback)
            except Exception as e:
                feedback = f"query failed: {str(e)[:200]}"
                print(f"ANALYST (try {attempt+1}) failed: {feedback}")
                continue
            verdict = reviewer(sq, sql, names, rows)
            print(f"ANALYST  -> {sq}\n  SQL: {' '.join(sql.split())}\nREVIEWER -> {verdict}\n")
            if verdict.upper().startswith("OK") or attempt == 1:
                findings.append({"question": sq, "columns": names, "rows": rows[:20],
                                 "reviewer": verdict})
                break
            feedback = verdict
    print("WRITER ->")
    print(llm(
        "You are the writer on a data team, explaining findings to a non-technical business "
        "client. Use plain English, lead with the answer, quote key numbers in USD, keep it "
        "under 150 words, and never invent numbers that are not in the findings. If a finding "
        "was not approved by the reviewer, say it is less certain.",
        f"Client question: {question}\nFindings: {json.dumps(findings, default=str)}\n"
        f"Data notes: {DATA_NOTES}", max_tokens=500))

team("How did revenue change from 2024 to 2025, and which category drove the change?")